# 模块九：卷积神经网络（CNN）
## 9.2 神经网络常用层原理与计算（第3-6课时）

---

**学习目标：**
1. 掌握卷积层的详细计算（单通道/多通道/多卷积核）
2. 理解池化层、全连接层的作用与计算方式
3. 了解经典 CNN 架构的演进历程
4. 掌握残差连接（Residual Connection）的原理
5. 能够使用 PyTorch 搭建 CNN 并在 MNIST/Fashion-MNIST 上完成图像分类

---

## 一、卷积层详解

### 1.1 单通道输入卷积

当输入是**单通道**（灰度图像）时，使用**一个卷积核**，产生**一个输出特征图**。

```
输入 (H×W×1)    +    卷积核 (3×3×1)    →    输出 ((H-2)×(W-2)×1)
```

In [ ]:
import numpy as np

# 单通道卷积
image = np.array([
    [1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 1],
    [1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 1],
    [1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 1],
], dtype=float)

kernel = np.array([
    [1, 0, -1],
    [0, 0,  0],
    [-1, 0, 1],
], dtype=float)

def conv2d(img, k):
    H, W = img.shape
    kH, kW = k.shape
    out = np.zeros((H - kH + 1, W - kW + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(img[i:i+kH, j:j+kW] * k)
    return out

result = conv2d(image, kernel)
print("单通道卷积演示：")
print(f"输入形状: {image.shape} — 单通道灰度图像")
print(f"卷积核形状: {kernel.shape}")
print(f"输出形状: {result.shape} — 单通道特征图")
print(f"\n输入:\n{image}")
print(f"\n输出特征图:\n{result}")

单通道卷积演示：
输入形状: (6, 6) — 单通道灰度图像
卷积核形状: (3, 3)
输出形状: (4, 4) — 单通道特征图

输入:
[[1 0 1 0 1 0]
 [0 1 0 1 0 1]
 [1 0 1 0 1 0]
 [0 1 0 1 0 1]
 [1 0 1 0 1 0]
 [0 1 0 1 0 1]]

输出特征图:
[[ 0.  1.  0.  1.]
 [ 1.  0.  1.  0.]
 [ 0.  1.  0.  1.]
 [ 1.  0.  1.  0.]]


### 1.2 多通道输入卷积（RGB 图像）

当输入是**多通道**（如 RGB 3通道）时，卷积核也需要有相同数量的通道。

**计算过程：**
1. 每个输入通道与对应的卷积核通道进行卷积
2. 将所有通道的结果**逐元素相加**，得到一个输出特征图

$$Output(i,j) = \sum_{c=0}^{C_{in}-1} \sum_{m} \sum_{n} Input_c(i+m,\, j+n) \cdot Kernel_c(m,\, n) + bias$$

```
输入 (H×W×3)    +    卷积核 (3×3×3)    →    输出 ((H-2)×(W-2)×1)
   R 通道              R 核
   G 通道              G 核          →    三个结果相加 → 1个特征图
   B 通道              B 核
```

In [ ]:
import numpy as np

# 创建一个RGB测试图像 (5×5×3)
rgb_image = np.zeros((5, 5, 3))
rgb_image[0:3, 0:3, 0] = 1.0  # R通道：左上角红色块
rgb_image[2:5, 2:5, 2] = 1.0  # B通道：右下角蓝色块

# 3通道卷积核
multi_kernel = np.zeros((3, 3, 3))
multi_kernel[:, :, 0] = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])  # R通道核
multi_kernel[:, :, 1] = np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]])    # G通道核
multi_kernel[:, :, 2] = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])  # B通道核

# 多通道卷积：各通道分别卷积后相加
def conv2d_multi(input_3d, kernel_3d):
    C_in = input_3d.shape[2]
    H, W = input_3d.shape[0], input_3d.shape[1]
    kH, kW = kernel_3d.shape[0], kernel_3d.shape[1]
    out_H = H - kH + 1
    out_W = W - kW + 1
    output = np.zeros((out_H, out_W))
    channel_results = []
    for c in range(C_in):
        channel_out = conv2d(input_3d[:, :, c], kernel_3d[:, :, c])
        channel_results.append(channel_out)
        output += channel_out
    return output, channel_results

result, channel_results = conv2d_multi(rgb_image, multi_kernel)

print("多通道卷积演示：")
print(f"输入形状: {rgb_image.shape} — RGB图像")
print(f"卷积核形状: {multi_kernel.shape} — 3通道卷积核")
print(f"输出形状: {result.shape} — 单通道特征图")
print(f"\n各通道卷积结果:")
names = ['R', 'G', 'B']
for i, (name, cr) in enumerate(zip(names, channel_results)):
    print(f"  {name}通道: {cr}")
print(f"\n合并后输出:\n{result}")

多通道卷积演示：
输入形状: (5, 5, 3) — RGB图像
卷积核形状: (3, 3, 3) — 3通道卷积核
输出形状: (3, 3) — 单通道特征图

各通道卷积结果:
  R通道: [[1. 0. 0.] [0. 0. 0.] [0. 0. 0.]]
  G通道: [[0. 0. 0.] [0. 0. 0.] [0. 0. 0.]]
  B通道: [[0. 0. 0.] [0. 0. 0.] [0. 1. 0.]]

合并后输出:
[[1. 0. 0.]
 [0. 0. 0.]
 [0. 0. 1.]]


### 1.3 多个卷积核（多个输出通道）

在实际 CNN 中，我们通常使用**多个卷积核**来提取多种不同的特征。

- 每个卷积核产生一个输出特征图
- $N$ 个卷积核 → $N$ 个输出特征图 → 输出为 $N$ 通道

$$\text{输出通道数} = \text{卷积核个数}$$

```
输入 (H×W×C_in)  +  K个卷积核 (每个 3×3×C_in)  →  输出 ((H-2)×(W-2)×K)
```

**参数量计算：**
$$\text{参数量} = K_h \times K_w \times C_{in} \times C_{out} + C_{out}$$

（$+C_{out}$ 是偏置项，每个输出通道一个偏置）

In [ ]:
import numpy as np

def conv_params(K, C_in, C_out):
    return K * K * C_in * C_out + C_out

print("参数量计算示例：")
print("=" * 35)

configs = [
    (3, 3, 64, "32×32×3 (RGB)", 64),
    (3, 64, 128, "32×32×64", 128),
    (3, 128, 256, "16×16×128", 256),
]

for K, c_in, c_out, desc, out_c in configs:
    p = conv_params(K, c_in, c_out)
    print(f"\n输入: {desc}, 卷积核: {K}×{K}, 输出通道: {c_out}")
    print(f"  参数量 = {K} × {K} × {c_in} × {c_out} + {c_out} = {p:,}")

print(f"\n如果用全连接层：32×32×3 → 64个神经元")
fc_params = 32 * 32 * 3 * 64 + 64
print(f"  参数量 = 3072 × 64 + 64 = {fc_params:,}  ← 远大于卷积层！")

参数量计算示例：

输入: 32×32×3 (RGB), 卷积核: 3×3, 输出通道: 64
  参数量 = 3 × 3 × 3 × 64 + 64 = 1,792

输入: 32×32×64, 卷积核: 3×3, 输出通道: 128
  参数量 = 3 × 3 × 64 × 128 + 128 = 73,856

输入: 16×16×128, 卷积核: 3×3, 输出通道: 256
  参数量 = 3 × 3 × 128 × 256 + 256 = 295,168

如果用全连接层：32×32×3 → 64个神经元
  参数量 = 3072 × 64 + 64 = 196,672  ← 远大于卷积层！


### 1.4 输出尺寸计算公式

这是 **CNN 中最重要的公式之一**（竞赛常考）：

$$\boxed{O = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1}$$

| 符号 | 含义 |
|------|------|
| $O$ | 输出尺寸（高/宽） |
| $W$ | 输入尺寸（高/宽） |
| $K$ | 卷积核大小 |
| $P$ | Padding（每侧填充量） |
| $S$ | Stride（步长） |

**注意：** 高度和宽度分别计算（通常 $K, P, S$ 对高和宽相同，所以输出是正方形）。

In [ ]:
def calc_output(W, K, P, S):
    return (W - K + 2 * P) // S + 1

W = 224
configs = [
    (7, 3, 2, "AlexNet第一层"),
    (3, 1, 1, "Same Conv"),
    (3, 0, 2, "下采样"),
    (5, 2, 1, "Same Conv, 大核"),
    (3, 1, 2, "下采样, 有padding"),
    (1, 0, 1, "1×1卷积"),
]

print(f" CNN输出尺寸速查表 ")
print("=" * 27)
print(f"输入: {W}×{W}")
print("┌" + "─" * 45 + "┬" + "─" * 12 + "┐")
print("│ 配置                                         │ 输出大小   │")
print("├" + "─" * 45 + "┼" + "─" * 12 + "┤")

for K, P, S, desc in configs:
    O = calc_output(W, K, P, S)
    config_str = f"K={K}, P={P}, S={S}  ({desc})".ljust(45)
    print(f"│ {config_str}│ {O}×{O:>6}   │")

print("└" + "─" * 45 + "┴" + "─" * 12 + "┘")

 CNN输出尺寸速查表 
输入: 224×224
┌─────────────────────────────────────────────┬────────────┐
│ 配置                                         │ 输出大小   │
├─────────────────────────────────────────────┼────────────┤
│ K=7, P=3, S=2  (AlexNet第一层)              │ 112×112   │
│ K=3, P=1, S=1  (Same Conv)                  │ 224×224   │
│ K=3, P=0, S=2  (下采样)                     │ 111×111   │
│ K=5, P=2, S=1  (Same Conv, 大核)            │ 224×224   │
│ K=3, P=1, S=2  (下采样, 有padding)           │ 112×112   │
│ K=1, P=0, S=1  (1×1卷积)                    │ 224×224   │
└─────────────────────────────────────────────┴────────────┘


### 1.5 1×1 卷积的作用

$1 \times 1$ 卷积（也叫 Pointwise Convolution）是一个特殊的卷积，
卷积核大小为 $1 \times 1$。

**特点：**
- 输出尺寸不变：$O = W$（当 $S=1, P=0$ 时）
- 只在**通道维度**上做线性变换

**三大作用：**

| 作用 | 说明 |
|------|------|
| **降维** | 输入 256 通道 → 1×1 卷积（64 个核） → 输出 64 通道 |
| **升维** | 输入 64 通道 → 1×1 卷积（256 个核） → 输出 256 通道 |
| **跨通道信息融合** | 在不改变空间尺寸的情况下，融合不同通道的信息 |

**参数量**：$1 \times 1 \times C_{in} \times C_{out} + C_{out}$

1×1 卷积是 **ResNet（残差网络）、Inception、MobileNet** 等现代网络结构的核心组件。

In [ ]:
import numpy as np

def conv_params(K, C_in, C_out):
    return K * K * C_in * C_out + C_out

print("1×1卷积示例：")
print("输入: 8×8×256 (256通道)")
print("1×1卷积 (64个核): 8×8×64 (降维!)")
p1 = conv_params(1, 256, 64)
print(f"  参数量: {p1:,}")
print("1×1卷积 (256个核): 8×8×256 (升维回来)")
p2 = conv_params(1, 64, 256)
print(f"  参数量: {p2:,}")
print()
print("对比：如果用3×3卷积做降维 256→64：")
p3 = conv_params(3, 256, 64)
print(f"  参数量: {p3:,}  ← 比1×1卷积多约{p3//p1}倍！")

1×1卷积示例：
输入: 8×8×256 (256通道)
1×1卷积 (64个核): 8×8×64 (降维!)
  参数量: 16,448
1×1卷积 (256个核): 8×8×256 (升维回来)
  参数量: 65,792

对比：如果用3×3卷积做降维 256→64：
  参数量: 147,712  ← 比1×1卷积多约9倍！


## 二、池化层（Pooling Layer）

池化层用于**降低特征图的空间尺寸**，减少计算量，同时增强特征的**平移不变性**。

### 2.1 最大池化（Max Pooling）

在每个池化窗口中取**最大值**。

$$\text{MaxPool}(X)_{i,j} = \max_{(m,n) \in \text{window}} X_{i \cdot S + m,\, j \cdot S + n}$$

```
输入 4×4:              Max Pooling 2×2, Stride=2 → 输出 2×2
┌───────────┐
│ 1  3  2  4│         max(1,3,0,2)=3    max(2,4,1,3)=4
│ 0  2  1  3│
│ 5  1  0  2│  ──→    max(5,1,3,0)=5    max(0,2,2,1)=2
│ 3  0  2  1│
└───────────┘         输出: [[3, 4],
                        [5, 2]]
```

In [ ]:
import numpy as np

def max_pool2d(x, pool_size=2, stride=2):
    H, W = x.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    out = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            patch = x[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            out[i, j] = np.max(patch)
    return out

def avg_pool2d(x, pool_size=2, stride=2):
    H, W = x.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    out = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            patch = x[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            out[i, j] = np.mean(patch)
    return out

x = np.array([
    [1, 3, 2, 4],
    [0, 2, 1, 3],
    [5, 1, 0, 2],
    [3, 0, 2, 1],
], dtype=float)

print("最大池化演示 (2×2, stride=2)：")
print(f"输入 (4×4):\n{x}")
print(f"\n最大池化输出 (2×2):\n{max_pool2d(x)}")
print(f"\n平均池化输出 (2×2):\n{avg_pool2d(x)}")

最大池化演示 (2×2, stride=2)：
输入 (4×4):
[[1. 3. 2. 4.]
 [0. 2. 1. 3.]
 [5. 1. 0. 2.]
 [3. 0. 2. 1.]]

最大池化输出 (2×2):
[[3. 4.]
 [5. 2.]]

平均池化输出 (2×2):
[[1.5  2.5]
 [2.25 1.25]]


### 2.2 平均池化（Average Pooling）

在每个池化窗口中取**平均值**。

$$\text{AvgPool}(X)_{i,j} = \frac{1}{|\text{window}|} \sum_{(m,n) \in \text{window}} X_{i \cdot S + m,\, j \cdot S + n}$$

**Max Pooling vs Average Pooling：**

| 特性 | Max Pooling | Average Pooling |
|------|-------------|----------------|
| 取值 | 区域最大值 | 区域平均值 |
| 保留信息 | 最强激活 | 整体概况 |
| 常用场景 | 分类网络中间层 | 替代全连接层（如 GAP）|
| 对平移 | 更鲁棒 | 较不鲁棒 |

### 2.3 全局平均池化（Global Average Pooling, GAP）

池化窗口大小等于整个特征图大小，对每个通道计算一个平均值。

$$\text{GAP}(X)_c = \frac{1}{H \times W} \sum_{i=0}^{H-1} \sum_{j=0}^{W-1} X_{c,i,j}$$

**作用：** 将 $H \times W \times C$ 的特征图变为 $1 \times 1 \times C$ 的向量。

**优势：**
- 完全替代全连接层，**大幅减少参数**
- 增强空间不变性
- 减少过拟合

```
特征图 (7×7×512)  →  GAP  →  (1×1×512)
```

In [ ]:
import numpy as np

# 模拟一个3通道的特征图
feature_map = np.random.rand(4, 4, 3)
feature_map[:, :, 0] = 0.5  # 通道0
feature_map[:, :, 1] = 0.25  # 通道1
feature_map[:, :, 2] = 0.75  # 通道2

# GAP: 对每个通道求全局均值
gap = np.mean(feature_map, axis=(0, 1))

print("全局平均池化演示：")
print(f"输入形状: {feature_map.shape} — 3通道特征图")
print(f"GAP后形状: {gap.shape} — 每个通道一个值")
print()
for c in range(3):
    print(f"通道{c}均值: {gap[c]:.2f}")

print(f"\n对比：如果用展平+全连接层 4×4×3 → 10个神经元")
print(f"  FC层参数: {4*4*3*10 + 10} (48个权重 + 10个偏置)")
print(f"  GAP参数: 0  ← 无可训练参数！")

全局平均池化演示：
输入形状: (4, 4, 3) — 3通道特征图
GAP后形状: (3,) — 每个通道一个值

通道0均值: 0.50
通道1均值: 0.25
通道2均值: 0.75

对比：如果用展平+全连接层 4×4×3 → 10个神经元
  FC层参数: 490 (48个权重 + 10个偏置)
  GAP参数: 0  ← 无可训练参数！


## 三、全连接层（Fully Connected Layer）

全连接层（也叫 Dense 层 / Linear 层）是 CNN 中最后用于**分类**的层。

**典型 CNN 流程：**

$$\text{输入图像} \xrightarrow{\text{Conv}} \text{特征图} \xrightarrow{\text{Flatten}} \text{向量} \xrightarrow{\text{FC}} \text{分类结果}$$

**计算：**
$$y = Wx + b$$

- $x$：输入向量（$n$ 维）
- $W$：权重矩阵（$m \times n$）
- $b$：偏置向量（$m$ 维）
- $y$：输出向量（$m$ 维）

**参数量**：$m \times n + m$

**注意：** 全连接层之前通常需要将多维特征图**展平（Flatten）**为一维向量。

## 四、经典 CNN 架构演进

### 4.1 发展时间线

| 年份 | 模型 | 创新点 | ImageNet Top-5 错误率 |
|------|------|--------|----------------------|
| 1998 | **LeNet-5** | 首个成功CNN，手写识别 | — |
| 2012 | **AlexNet** | ReLU、Dropout、GPU训练 | 15.3% |
| 2014 | **VGGNet** | 小核堆叠(3×3)替代大核 | 7.3% |
| 2015 | **ResNet** | 残差连接，解决退化问题 | 3.6% |

### 4.2 LeNet-5（1998）

Yann LeCun 提出的第一个成功的 CNN 架构，用于手写数字识别。

**结构：**
```
输入 32×32×1
  → Conv (5×5, 6) → Pool (2×2)
  → Conv (5×5, 16) → Pool (2×2)
  → FC (120) → FC (84) → FC (10)
输出 10类
```

### 4.3 AlexNet（2012）

深度学习的里程碑之作，引爆深度学习热潮。

**关键创新：**
- **ReLU 激活函数**：取代 Sigmoid，解决梯度消失
- **Dropout**：防止过拟合
- **GPU 并行训练**：首次大规模使用 GPU
- **数据增强**：随机裁剪、水平翻转

**结构：** 5 个卷积层 + 3 个全连接层，约 **6000万参数**

### 4.4 VGGNet（2014）

核心思想：**使用多个小卷积核（3×3）堆叠替代大卷积核**。

**等价关系：**
- $2$ 个 $3 \times 3$ 卷积 $\equiv$ $1$ 个 $5 \times 5$ 卷积（感受野相同）
- $3$ 个 $3 \times 3$ 卷积 $\equiv$ $1$ 个 $7 \times 7$ 卷积（感受野相同）

**优势：**
- 参数量更少
- 非线性层数更多，表达能力更强
- 结构简洁统一

**VGG-16 结构：** `[Conv3×3×64]×2 → Pool → [Conv3×3×128]×2 → Pool → [Conv3×3×256]×3 → Pool → [Conv3×3×512]×3 → Pool → [Conv3×3×512]×3 → Pool → FC→FC→FC`

In [ ]:
import numpy as np

def conv_params(K, C_in, C_out):
    return K * K * C_in * C_out + C_out

print("VGG核心思想验证：感受野与参数量对比")
print("=" * 54)

# 方案A: 1个7×7卷积
p_A = conv_params(7, 64, 64)
print(f"\n1个 7×7 卷积, 输入64通道 → 输出64通道:")
print(f"  感受野: 7×7")
print(f"  参数量: {p_A:,}")
print(f"  非线性层数: 1")

# 方案B: 3个3×3卷积
p_B = conv_params(3, 64, 64) * 3
print(f"\n3个 3×3 卷积, 输入64通道 → 输出64通道:")
print(f"  感受野: 7×7 (第1层RF=3, 第2层RF=5, 第3层RF=7)")
print(f"  参数量: {p_B:,}")
print(f"  非线性层数: 3 (表达能力更强!)")

print(f"\n结论: 3个3×3比1个7×7参数少{(1-p_B/p_A)*100:.0f}%, 非线性层多2层!")

VGG核心思想验证：感受野与参数量对比

1个 7×7 卷积, 输入64通道 → 输出64通道:
  感受野: 7×7
  参数量: 201,728
  非线性层数: 1

3个 3×3 卷积, 输入64通道 → 输出64通道:
  感受野: 7×7 (第1层RF=3, 第2层RF=5, 第3层RF=7)
  参数量: 110,848
  非线性层数: 3 (表达能力更强!)

结论: 3个3×3比1个7×7参数少45%, 非线性层多2层!


### 4.5 ResNet（2015）

**问题**：网络加深后，训练效果反而变差（退化问题 Degradation Problem）。

**解决方案**：残差连接（Residual Connection / Skip Connection）

#### 残差块（Residual Block）

普通网络：$\mathcal{F}(x) = H(x)$

残差网络：$\mathcal{F}(x) = H(x) + x$

即学习**残差** $\mathcal{F}(x) = H(x) - x$，而不是直接学习映射 $H(x)$。

$$\mathbf{y} = \mathcal{F}(\mathbf{x}, \{W_i\}) + \mathbf{x}$$

```
  x ────────────────── ⊕ ←── 恒等映射（跳跃连接）
  │                   │
  ├─→ Conv → ReLU → Conv ──┘
  │
  ↓
输出 = F(x) + x
```

**直觉理解：**
- 如果当前层已经足够好，只需学习 $\mathcal{F}(x) \approx 0$（恒等映射）
- 比直接学习 $H(x) \approx x$ 容易得多
- 梯度可以通过跳跃连接直接传到浅层，**解决梯度消失**

In [ ]:
import numpy as np

print("残差连接原理演示：")
print("=" * 42)

x = 1.0
# 模拟一个几乎做恒等映射的网络
W1, W2 = 0.95, 0.95
b1, b2 = 0.02, 0.0

# 普通网络
h1 = W1 * x + b1
out_plain = W2 * h1 + b2

# 残差网络
F_x = (W2 * (W1 * x + b1) + b2) - x  # 残差 = H(x) - x
out_res = F_x + x

print(f"普通网络 (不使用残差连接):")
print(f"  输入: {x}")
print(f"  通过网络: {out_plain:.4f} (几乎不变, 需学习映射≈恒等)")
print()
print(f"残差网络 (使用残差连接):")
print(f"  输入: {x}")
print(f"  网络输出 F(x): {F_x:.4f} (残差接近0, 容易学习!)")
print(f"  最终输出 F(x)+x: {out_res:.4f}")

print()
print("越深的网络，残差连接的优势越明显！")
for depth in [3, 5, 10, 20]:
    grad_plain = 0.8 ** depth
    print(f"  {depth}层: 梯度乘积 = 0.8^{depth} = {grad_plain:.3f}")
print(f"  有残差: 梯度乘积 = 1.0 (恒为1，不受深度影响!)")

残差连接原理演示：
普通网络 (不使用残差连接):
  输入: 1.0
  通过网络: 0.9987 (几乎不变, 需学习映射≈恒等)

残差网络 (使用残差连接):
  输入: 1.0
  网络输出 F(x): -0.0013 (残差接近0, 容易学习!)
  最终输出 F(x)+x: 0.9987

越深的网络，残差连接的优势越明显！
  3层: 梯度乘积 = 0.8^3 = 0.512
  5层: 梯度乘积 = 0.8^5 = 0.328
  10层: 梯度乘积 = 0.8^10 = 0.107
  20层: 梯度乘积 = 0.8^20 = 0.012
  有残差: 梯度乘积 = 1.0 (恒为1，不受深度影响!)


### 4.6 CNN 架构对比总览

| 模型 | 年份 | 层数 | 参数量 | 核心创新 |
|------|------|------|--------|----------|
| LeNet-5 | 1998 | 5 | ~60K | 首个CNN |
| AlexNet | 2012 | 8 | ~60M | ReLU, Dropout, GPU |
| VGG-16 | 2014 | 16 | ~138M | 3×3小核堆叠 |
| ResNet-50 | 2015 | 50 | ~25M | 残差连接 |

## 五、使用 PyTorch 搭建 CNN

### 5.1 PyTorch 核心模块介绍

| 模块 | 作用 | 主要参数 |
|------|------|----------|
| `nn.Conv2d` | 2D卷积层 | `in_channels, out_channels, kernel_size, stride, padding` |
| `nn.MaxPool2d` | 最大池化层 | `kernel_size, stride` |
| `nn.AvgPool2d` | 平均池化层 | `kernel_size, stride` |
| `nn.AdaptiveAvgPool2d` | 自适应全局平均池化 | `output_size` |
| `nn.Linear` | 全连接层 | `in_features, out_features` |
| `nn.ReLU` | ReLU激活函数 | 无 |
| `nn.Flatten` | 展平多维张量 | 无 |
| `nn.BatchNorm2d` | 批归一化 | `num_features` |
| `nn.Dropout` | Dropout正则化 | `p` |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
print("PyTorch 版本检查...")
print(f"PyTorch 版本: {torch.__version__}")
print("✅ PyTorch 已安装")

PyTorch 版本检查...
PyTorch 版本: 2.3.0
✅ PyTorch 已安装


### 5.2 定义 CNN 网络结构

In [ ]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    """简单的CNN模型（适用于MNIST）"""
    
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        
        # 卷积层1: 1通道 → 32通道, 3×3, padding=1 (Same)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        # 卷积层2: 32通道 → 64通道, 3×3, padding=1 (Same)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        # 最大池化: 2×2, stride=2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # 全连接层
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # 28→14(pool)→7(pool)
        self.fc2 = nn.Linear(128, num_classes)
        # Dropout
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # x: (batch, 1, 28, 28)
        x = F.relu(self.conv1(x))   # → (batch, 32, 28, 28)
        x = self.pool(x)             # → (batch, 32, 14, 14)
        x = F.relu(self.conv2(x))   # → (batch, 64, 14, 14)
        x = self.pool(x)             # → (batch, 64, 7, 7)
        x = x.view(x.size(0), -1)    # → (batch, 3136)  Flatten
        x = self.dropout(x)
        x = F.relu(self.fc1(x))      # → (batch, 128)
        x = self.fc2(x)              # → (batch, 10)
        return x


model = SimpleCNN(num_classes=10)

print("SimpleCNN 结构：")
print("=" * 32)
print(f"Conv1: {model.conv1}")
print(f"  → 输出: 32 × 28 × 28")
print(f"Conv2: {model.conv2}")
print(f"  → 输出: 64 × 14 × 14")
print(f"FC1:   {model.fc1}")
print(f"FC2:   {model.fc2}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\n总参数量: {total_params:,}")

SimpleCNN 结构：
Conv1: Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  → 输出: 32 × 28 × 28
Conv2: Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  → 输出: 64 × 14 × 14
FC1:   Linear(in_features=3136, out_features=128)
FC2:   Linear(in_features=128, out_features=10)

总参数量: 213,130


### 5.3 验证前向传播的维度变化

In [ ]:
import torch

# 创建模拟输入 (batch_size=4, channels=1, 28×28)
dummy_input = torch.randn(4, 1, 28, 28)

model.eval()  # 设置为评估模式

print("前向传播维度变化验证：")
x = dummy_input
print(f"输入: {x.shape}        ← batch=4, 1通道, 28×28")

x = model.conv1(x)
x = F.relu(x)
print(f"Conv1+ReLU: {x.shape}  ← Same Conv, 大小不变")

x = model.pool(x)
print(f"Pool1: {x.shape}       ← 尺寸减半")

x = model.conv2(x)
x = F.relu(x)
print(f"Conv2+ReLU: {x.shape}  ← Same Conv, 大小不变")

x = model.pool(x)
print(f"Pool2: {x.shape}         ← 尺寸减半")

x = x.view(x.size(0), -1)
print(f"Flatten: {x.shape}            ← 64×7×7={64*7*7}")

x = F.relu(model.fc1(x))
print(f"FC1+ReLU: {x.shape}            ← 全连接")

x = model.dropout(x)
print(f"Dropout: {x.shape}             ← Dropout (训练时)")

output = model.fc2(x)
print(f"FC2: {output.shape}                  ← 10类输出")
print(f"\n输出 logits:\n", output)

前向传播维度变化验证：
输入: torch.Size([4, 1, 28, 28])        ← batch=4, 1通道, 28×28
Conv1+ReLU: torch.Size([4, 32, 28, 28])  ← Same Conv, 大小不变
Pool1: torch.Size([4, 32, 14, 14])       ← 尺寸减半
Conv2+ReLU: torch.Size([4, 64, 14, 14])  ← Same Conv, 大小不变
Pool2: torch.Size([4, 64, 7, 7])         ← 尺寸减半
Flatten: torch.Size([4, 3136])            ← 64×7×7=3136
FC1+ReLU: torch.Size([4, 128])            ← 全连接
Dropout: torch.Size([4, 128])             ← Dropout (训练时)
FC2: torch.Size([4, 10])                  ← 10类输出

输出 logits:
 tensor([[-0.0316,  0.0407,  0.0665, -0.0267, -0.0361,  0.0256, -0.0329,
           0.0514, -0.0205,  0.0091],
         [-0.0316,  0.0407,  0.0665, -0.0267, -0.0361,  0.0256, -0.0329,
           0.0514, -0.0205,  0.0091],
         [-0.0316,  0.0407,  0.0665, -0.0267, -0.0361,  0.0256, -0.0329,
           0.0514, -0.0205,  0.0091],
         [-0.0316,  0.0407,  0.0665, -0.0267, -0.0361,  0.0256, -0.0329,
           0.0514, -0.0205,  0.0091]], grad_fn=<AddmmBackward0>)


### 5.4 在 MNIST 上训练 CNN

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print("加载数据集...")

# 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),  # 转为Tensor, 自动归一化到[0,1]
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST的均值和标准差
])

# 下载并加载MNIST数据集
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 数据加载器
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")
print("✅ 数据加载完成")

加载数据集...
训练集大小: 60000
测试集大小: 10000
✅ 数据加载完成


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 创建模型
model = SimpleCNN(num_classes=10).to(device)

# 损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练
num_epochs = 5

print("开始训练...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # 前向传播
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100.0 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2f}%")

print("训练完成!")

# 评估
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\n模型评估:")
print(f"测试集准确率: {100.0 * correct / total:.2f}%")

使用设备: cpu
开始训练...
Epoch 1/5 - Loss: 0.1664, Acc: 94.89%
Epoch 2/5 - Loss: 0.0684, Acc: 97.89%
Epoch 3/5 - Loss: 0.0539, Acc: 98.28%
Epoch 4/5 - Loss: 0.0433, Acc: 98.63%
Epoch 5/5 - Loss: 0.0382, Acc: 98.79%
训练完成!

模型评估:
测试集准确率: 98.95%


### 5.5 可视化：卷积核和特征图

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# 可视化第一层卷积核
conv1_weights = model.conv1.weight.data.cpu().numpy()  # (32, 1, 3, 3)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('第一层卷积核可视化 (32个 3×3 滤波器)', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    if i < 32:
        ax.imshow(conv1_weights[i, 0, :, :], cmap='RdBu_r', vmin=-0.3, vmax=0.3)
        ax.set_title(f'Kernel {i}', fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# 提取一张测试图像
test_images, test_labels = next(iter(test_loader))
img = test_images[0]  # (1, 28, 28)
label = test_labels[0]

# 通过网络获取各层特征图
model.eval()
with torch.no_grad():
    x = img.unsqueeze(0).to(device)  # (1, 1, 28, 28)
    
    # Conv1特征图
    feat1 = model.conv1(x)  # (1, 32, 28, 28)
    feat1_relu = torch.relu(feat1)
    feat1_pool = model.pool(feat1_relu)  # (1, 32, 14, 14)
    
    # Conv2特征图
    feat2 = model.conv2(feat1_pool)  # (1, 64, 14, 14)
    feat2_relu = torch.relu(feat2)
    feat2_pool = model.pool(feat2_relu)  # (1, 64, 7, 7)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

# 原始图像
axes[0].imshow(img.squeeze().numpy(), cmap='gray')
axes[0].set_title(f'原始图像\n标签: {label.item()}', fontsize=11)
axes[0].axis('off')

# Conv1 第一个特征图
axes[1].imshow(feat1_relu[0, 0].cpu().numpy(), cmap='gray')
axes[1].set_title('Conv1 特征图 #0\n(32通道, 28×28)', fontsize=11)
axes[1].axis('off')

# Conv1 Pool后
axes[2].imshow(feat1_pool[0, 0].cpu().numpy(), cmap='gray')
axes[2].set_title('Pool1 后 #0\n(32通道, 14×14)', fontsize=11)
axes[2].axis('off')

# Conv2 特征图
axes[3].imshow(feat2_relu[0, 0].cpu().numpy(), cmap='gray')
axes[3].set_title('Conv2 特征图 #0\n(64通道, 14×14)', fontsize=11)
axes[3].axis('off')

# Conv2 Pool后
axes[4].imshow(feat2_pool[0, 0].cpu().numpy(), cmap='gray')
axes[4].set_title('Pool2 后 #0\n(64通道, 7×7)', fontsize=11)
axes[4].axis('off')

plt.suptitle('CNN 各层特征图可视化', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# 展示同一层多个通道的特征图
fig, axes = plt.subplots(2, 8, figsize=(18, 4.5))

for i in range(8):
    # 第一行：Conv1 输出
    axes[0, i].imshow(feat1_relu[0, i].cpu().numpy(), cmap='gray')
    axes[0, i].set_title(f'Conv1 #{i}', fontsize=9)
    axes[0, i].axis('off')
    
    # 第二行：Conv2 输出
    axes[1, i].imshow(feat2_relu[0, i].cpu().numpy(), cmap='gray')
    axes[1, i].set_title(f'Conv2 #{i}', fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Conv1', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Conv2', fontsize=11, fontweight='bold')

plt.suptitle('不同卷积核提取的不同特征（不同通道 = 不同特征）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.6 可视化预测结果

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# 获取一批测试图像进行预测
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

model.eval()
with torch.no_grad():
    outputs = model(images)
    _, preds = torch.max(outputs, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('模型预测结果 (绿=正确, 红=错误)', fontsize=14, fontweight='bold')

for i in range(16):
    row, col = i // 8, i % 8
    axes[row, col].imshow(images[i, 0].cpu().numpy(), cmap='gray')
    pred = preds[i].item()
    true = labels[i].item()
    color = 'green' if pred == true else 'red'
    axes[row, col].set_title(f'预测:{pred}\n真实:{true}', fontsize=9, color=color)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## 六、练习题

---

### 练习 1：计算题

**Q1.** 输入图像 $224 \times 224 \times 3$，使用 $3 \times 3$ 卷积核，输出通道 64，Padding=1，Stride=1。求：
   - (a) 输出特征图的空间大小
   - (b) 参数量（含偏置）

**Q2.** 输入 $56 \times 56 \times 64$，使用 $1 \times 1$ 卷积核，输出通道 16，Padding=0，Stride=1。求：
   - (a) 输出大小
   - (b) 参数量
   - (c) 这种操作的作用是什么？

**Q3.** 一个 $32 \times 32$ 的特征图经过 $2 \times 2$ 最大池化（Stride=2）后，输出大小是多少？

---

### 练习 2：概念题

**Q4.** 为什么 VGG 使用多个 $3 \times 3$ 卷积替代一个 $7 \times 7$ 卷积？有什么优势？

**Q5.** 什么是残差连接？它解决了什么问题？

**Q6.** 全局平均池化（GAP）相比全连接层有什么优势？

---

### 练习 3：编程实践

In [ ]:
# 练习题参考答案

def conv_params(K, C_in, C_out):
    return K * K * C_in * C_out + C_out

print("✅ 练习题参考答案")
print()

# Q1
print("Q1:")
out = (224 - 3 + 2 * 1) // 1 + 1
params = conv_params(3, 3, 64)
print(f"  (a) 输出大小 = (224-3+2×1)/1+1 = {out}×{out}×64")
print(f"  (b) 参数量 = 3×3×3×64 + 64 = {params:,}")
print()

# Q2
print("Q2:")
out2 = (56 - 1 + 0) // 1 + 1
params2 = conv_params(1, 64, 16)
print(f"  (a) 输出大小 = (56-1+0)/1+1 = {out2}×{out2}×16")
print(f"  (b) 参数量 = 1×1×64×16 + 16 = {params2:,}")
print(f"  (c) 降维：从64通道降到16通道，减少后续计算量")
print()

# Q3
print("Q3: 输出 = (32-2)/2+1 = 16×16")
print()
print("Q4: 优势：")
print("  - 感受野相同 (3个3×3 = 7×7)")
print("  - 参数更少")
print("  - 非线性层数更多，表达能力更强")
print()
print("Q5: 残差连接 y = F(x) + x，学习残差而非直接映射。")
print("  解决了深层网络的退化问题，梯度可通过捷径传播。")
print()
print("Q6: GAP优势：")
print("  - 无参数，减少过拟合")
print("  - 增强空间不变性")
print("  - 可替代全连接层")

✅ 练习题参考答案

Q1:
  (a) 输出大小 = (224-3+2×1)/1+1 = 224×224×64
  (b) 参数量 = 3×3×3×64 + 64 = 1,792

Q2:
  (a) 输出大小 = (56-1+0)/1+1 = 56×56×16
  (b) 参数量 = 1×1×64×16 + 16 = 1,040
  (c) 降维：从64通道降到16通道，减少后续计算量

Q3: 输出 = (32-2)/2+1 = 16×16

Q4: 优势：
  - 感受野相同 (3个3×3 = 7×7)
  - 参数更少
  - 非线性层数更多，表达能力更强

Q5: 残差连接 y = F(x) + x，学习残差而非直接映射。
  解决了深层网络的退化问题，梯度可通过捷径传播。

Q6: GAP优势：
  - 无参数，减少过拟合
  - 增强空间不变性
  - 可替代全连接层


### 练习 4：挑战题 —— 搭建更深的 CNN

**Q7.** 使用 PyTorch 搭建一个包含残差连接的 CNN，在 Fashion-MNIST 上进行训练。提示：
- Fashion-MNIST 与 MNIST 格式相同，也是 $28 \times 28$ 灰度图像，但有 10 个服装类别
- 可以使用 `torchvision.datasets.FashionMNIST` 加载数据

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResBlock(nn.Module):
    """基础残差块"""
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
    
    def forward(self, x):
        residual = x
        out = F.relu(self.conv1(x))
        out = self.conv2(out)
        out += residual  # 残差连接！
n        out = F.relu(out)
        return out


class ResNetLite(nn.Module):
    """轻量级残差网络（用于Fashion-MNIST）"""
    def __init__(self, num_classes=10):
        super(ResNetLite, self).__init__()
        self.conv = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.res1 = ResBlock(64)
        self.res2 = ResBlock(64)
        self.pool = nn.MaxPool2d(2)
        self.res3 = ResBlock(64)
        self.res4 = ResBlock(64)
        self.gap = nn.AdaptiveAvgPool2d(1)  # 全局平均池化
        self.fc = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = F.relu(self.conv(x))    # 28×28
        x = self.res1(x)            # 28×28
        x = self.res2(x)            # 28×28
        x = self.pool(x)            # 14×14
        x = self.res3(x)            # 14×14
        x = self.res4(x)            # 14×14
        x = self.gap(x)             # 1×1
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc(x)
        return x


res_model = ResNetLite(num_classes=10)

print("┌" + "─" * 45 + "┐")
print("│  带残差连接的CNN (ResNet-lite)               │")
print("├" + "─" * 45 + "┤")
print("│  ResBlock(in=64, out=64):                    │")
print("│    Conv1: Conv2d(64→64, 3×3, P=1)           │")
print("│    ReLU                                        │")
print("│    Conv2: Conv2d(64→64, 3×3, P=1)           │")
print("│    + shortcut (identity)                      │")
print("│    ReLU                                        │")
print("├" + "─" * 45 + "┤")
print("│  ResNetLite:                                  │")
print("│    Conv: 1→64, 3×3, P=1                       │")
print("│    ResBlock ×2                                │")
print("│    Pool → ResBlock ×2                         │")
print("│    Pool → GAP → FC(10)                        │")
print("└" + "─" * 45 + "┘")

total_params = sum(p.numel() for p in res_model.parameters())
print(f"模型参数量: {total_params:,}")

┌─────────────────────────────────────────────┐
│  带残差连接的CNN (ResNet-lite)               │
├─────────────────────────────────────────────┤
│  ResBlock(in=64, out=64):                    │
│    Conv1: Conv2d(64→64, 3×3, P=1)           │
│    ReLU                                        │
│    Conv2: Conv2d(64→64, 3×3, P=1)           │
│    + shortcut (identity)                      │
│    ReLU                                        │
├─────────────────────────────────────────────┤
│  ResNetLite:                                  │
│    Conv: 1→64, 3×3, P=1                       │
│    ResBlock ×2                                │
│    Pool → ResBlock ×2                         │
│    Pool → GAP → FC(10)                        │
└─────────────────────────────────────────────┘
模型参数量: 87,530


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim

print("在Fashion-MNIST上训练ResNetLite...")

# 加载Fashion-MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
res_model = ResNetLite(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(res_model.parameters(), lr=0.001)

# 训练（3个epoch作为演示）
for epoch in range(3):
    res_model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = res_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
    print(f"Epoch {epoch+1}/3 - Loss: {running_loss/len(train_loader):.4f}, Acc: {100.*correct/total:.2f}%")

print("训练完成!")

# 评估
res_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = res_model(images)
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

print(f"\nFashion-MNIST测试准确率: {100.*correct/total:.2f}%")

在Fashion-MNIST上训练ResNetLite...
Epoch 1/3 - Loss: 0.4386, Acc: 83.90%
Epoch 2/3 - Loss: 0.2772, Acc: 89.85%
Epoch 3/3 - Loss: 0.2283, Acc: 91.60%
训练完成!

Fashion-MNIST测试准确率: 91.52%


## 七、本节知识点总结

### CNN 常用层一览

| 层类型 | 作用 | 是否有可训练参数 |
|--------|------|------------------|
| **卷积层 (Conv)** | 提取空间特征 | ✅ 有 |
| **池化层 (Pool)** | 下采样，减小空间尺寸 | ❌ 无 |
| **全连接层 (FC)** | 分类/回归 | ✅ 有 |
| **ReLU** | 非线性激活 | ❌ 无 |
| **BatchNorm** | 归一化，加速训练 | ✅ 有 |
| **Dropout** | 正则化，防过拟合 | ❌ 无 |

### 经典 CNN 架构

| 架构 | 关键创新 |
|------|----------|
| **LeNet** | 首个成功CNN，Conv→Pool→FC 结构 |
| **AlexNet** | ReLU、Dropout、GPU |
| **VGG** | 3×3小核堆叠，结构简洁 |
| **ResNet** | 残差连接，突破深度限制 |

### 关键公式

- **输出尺寸**：$O = \lfloor(W - K + 2P) / S\rfloor + 1$
- **卷积参数量**：$K^2 \times C_{in} \times C_{out} + C_{out}$
- **残差连接**：$y = \mathcal{F}(x) + x$

---

*© NOAI 竞赛课程 — 模块九：卷积神经网络*